# HolmesVAU description on UCF-Crime

Switch `scorer_type` between `'urdmu'` (default HolmesVAU ATS) and `'vadclip'` (VadCLIP precomputed features) to compare.

In [ ]:
import sys
from pathlib import Path

import torch
import matplotlib.pyplot as plt
from decord import VideoReader, cpu

# Make description_runner importable wherever this notebook lives
MODULE_DIR = Path('/workspace/VidAnomalyRetrieval/DescriptionModule')
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

from description_runner import load_pipeline, run_inference

In [ ]:
# === Paths (sửa cho khớp server) ===
MLLM_PATH            = MODULE_DIR / 'HolmesVAU' / 'ckpts' / 'HolmesVAU-2B'
URDMU_CKPT           = MODULE_DIR / 'Holmes-VAU-ATS' / 'anomaly_scorer.pth'
VADCLIP_CKPT         = MODULE_DIR / 'Org_vadclip' / 'model_ucf.pth'
VADCLIP_FEATURES_DIR = MODULE_DIR / 'UCFClipFeatures'
VIDEO_DIR            = Path('/workspace/VidAnomalyRetrieval/UCF_Video')
GT_FILE              = MODULE_DIR / 'VadCLIP' / 'list' / 'Temporal_Anomaly_Annotation.txt'

device = torch.device('cuda')

# === Choose scorer ===
SCORER_TYPE = 'vadclip'   # 'urdmu' | 'vadclip'

mllm, tokenizer, generation_config, scorer = load_pipeline(
    scorer_type=SCORER_TYPE,
    mllm_path=str(MLLM_PATH),
    urdmu_ckpt=str(URDMU_CKPT),
    vadclip_ckpt=str(VADCLIP_CKPT),
    vadclip_features_dir=str(VADCLIP_FEATURES_DIR),
    device=device,
)
print(f'Loaded MLLM + scorer={scorer.name}')

In [ ]:
# === Ground truth lookup ===
def load_gt(gt_file: Path) -> dict:
    """Parse Temporal_Anomaly_Annotation.txt -> {video_stem: (category, [(s1,e1), (s2,e2)?])}."""
    table = {}
    for line in Path(gt_file).read_text().splitlines():
        toks = line.split()
        if len(toks) < 6:
            continue
        stem, cat = toks[0], toks[1]
        s1, e1, s2, e2 = map(int, toks[2:6])
        segs = [(s1, e1)] if s1 != -1 else []
        if s2 != -1:
            segs.append((s2, e2))
        table[stem] = (cat, segs)
    return table

GT = load_gt(GT_FILE)
print(f'Loaded {len(GT)} GT entries. Example: Abuse028_x264 ->', GT.get('Abuse028_x264'))

In [ ]:
# === Pick video + run ===
VIDEO_NAME = 'Abuse028_x264'           # đổi tên video ở đây
video_path = VIDEO_DIR / f'{VIDEO_NAME}.mp4'
prompt = 'Could you specify the anomaly events present in the video?'

# Print ground truth before inference
if VIDEO_NAME in GT:
    cat, segs = GT[VIDEO_NAME]
    seg_str = ', '.join(f'[{s}-{e}]' for s, e in segs) if segs else '(no anomaly segment)'
    print(f'GT category : {cat}')
    print(f'GT segments : {seg_str}  (frame indices)')
else:
    print(f'No GT entry for {VIDEO_NAME} (likely a Normal video)')

pred, frame_indices, anomaly_score = run_inference(
    str(video_path), prompt, mllm, tokenizer, generation_config, scorer,
    select_frames=12, use_ATS=True,
)
print('\nUser     :', prompt)
print('HolmesVAU:', pred)

In [ ]:
# === Visualize sampled frames ===
import numpy as np

vr = VideoReader(str(video_path), ctx=cpu(0), num_threads=1)
frames = [vr[i].asnumpy() for i in frame_indices]
h, w, _ = frames[0].shape
strip = np.zeros((h, w * len(frames), 3), dtype=np.uint8)
for i, f in enumerate(frames):
    strip[:, i*w:(i+1)*w] = f
    strip[:, i*w:i*w+5] = 255
plt.figure(figsize=(20, 5))
plt.imshow(strip); plt.axis('off')
plt.title(f'Sampled frames (scorer={scorer.name})')
plt.show()

In [ ]:
# === Anomaly score curve + GT overlay ===
if anomaly_score is None:
    print('use_ATS=False or video too short — no anomaly score to plot.')
else:
    DENSE_STRIDE = 16   # must match dense_sample_freq in run_inference
    plt.figure(figsize=(10, 2.5))
    x = np.arange(len(anomaly_score)) * DENSE_STRIDE   # convert snippet -> frame index
    plt.plot(x, anomaly_score, label=f'{scorer.name} score', color='C0')
    # selected frames
    for fi in frame_indices:
        plt.vlines(fi, 0, 1, colors='C3', alpha=0.5)
    # GT segments
    if VIDEO_NAME in GT:
        for s, e in GT[VIDEO_NAME][1]:
            plt.axvspan(s, e, color='C2', alpha=0.2, label='GT anomaly')
    plt.ylim(0, 1)
    plt.xlabel('frame index')
    plt.ylabel('anomaly score')
    plt.title(f'{VIDEO_NAME}  ({scorer.name})')
    handles, labels = plt.gca().get_legend_handles_labels()
    by_label = dict(zip(labels, handles))   # dedupe legend
    plt.legend(by_label.values(), by_label.keys(), loc='upper right')
    plt.tight_layout()
    plt.show()